##  **1. Mô tả bài toán**

Có **N khách hàng** (1, 2, …, N) cần được bảo trì mạng internet. Khách hàng (i) ở địa điểm (i).

* Thời gian bảo trì tại khách hàng (i): (d(i))
* Có **K nhân viên kỹ thuật**, xuất phát từ **depot (điểm 0)** tại thời điểm (t_0 = 0)
* Thời gian di chuyển giữa các điểm: (t(i,j) \le 10{,}000)

Một lộ trình của nhân viên (k) được biểu diễn:

```
r[0], r[1], r[2], ..., r[Lk]
```

Trong đó:

```
r[0] = r[Lk] = 0 (depot)
```

---

## **Mục tiêu**

Tối thiểu hóa:

$
 \max_{k=1..K} Worktime(k)
$

Trong đó:

$$
Worktime(k) = TravelTime(k) + ServiceTime(k)
$$

=> Ý nghĩa:
**Nhân viên bận nhất phải bận ít nhất có thể**

---

## **Input**

```
Line 1: N K (1 ≤ N ≤ 1000, 1 ≤ K ≤ 100)

Line 2: d(1), d(2), ..., d(N)

Next N+1 lines: ma trận thời gian t(i,j)
```

### Ví dụ Input

```
5 2

60 80 70 10 90
0 50 100 60 40 80
50 0 50 40 20 60
100 50 0 50 70 40
60 40 50 0 60 20
40 20 70 60 0 80
80 60 40 20 80 0
```

---

##  **Output**

```
Line 1: K

For each k:
  Line 2k: Lk
  Line 2k+1: route
```

### 🔹 Ví dụ Output

```
2
5
0 5 3 4 0
4
0 2 1 0
```

---

#  **2. Thuật toán Genetic Algorithm (GA)**

##  Ý tưởng chính

Thay vì tìm một nghiệm duy nhất, ta:

* Duy trì **một quần thể nghiệm**
* Tiến hóa qua nhiều thế hệ
* Chọn nghiệm tốt dần theo **fitness**

---

##  **3. Biểu diễn nghiệm (Encoding)**

Mỗi cá thể là một **hoán vị khách hàng**:

```
[1, 5, 3, 4, 2]
```

 Sau đó chia thành K route:

```
Route 1: [1, 3, 2]
Route 2: [5, 4]
```

---

##  **4. Hàm đánh giá (Fitness Function)**

```
fitness = max workload của K nhân viên
```

Trong đó:

```
workload = tổng thời gian di chuyển + bảo trì
```

 Mục tiêu: **minimize fitness**

---

##  **5. Các bước thuật toán**

###  Bước 1: Khởi tạo quần thể

* Sinh ngẫu nhiên nhiều nghiệm (permutation)

---

###  Bước 2: Đánh giá

* Tính fitness cho từng cá thể

---

###  Bước 3: Selection (Chọn lọc)

* Dùng **Tournament Selection**
* Chọn cá thể tốt hơn

---

###  Bước 4: Crossover (Lai ghép)

* Dùng **Order Crossover (OX)**
* Giữ thứ tự khách hàng

---

###  Bước 5: Mutation (Đột biến)

* Hoán đổi 2 khách hàng bất kỳ

---

###  Bước 6: Lặp

* Lặp qua nhiều thế hệ
* Cập nhật nghiệm tốt nhất




In [ ]:
import random
import math


In [ ]:
# =========================
# 3. Encoding + Decode
# =========================
def decode(chromosome, K, d, t):
    """
    Chuyển permutation → K routes
    (gán greedy để cân bằng workload)
    """
    routes = [[] for _ in range(K)]
    workloads = [0]*K

    for customer in chromosome:
        best_k = None
        best_cost = float('inf')

        for k in range(K):
            prev = 0 if not routes[k] else routes[k][-1]
            cost = workloads[k] + t[prev][customer] + d[customer]

            if cost < best_cost:
                best_cost = cost
                best_k = k

        routes[best_k].append(customer)
        workloads[best_k] = best_cost

    return routes


# =========================
# Compute workload
# =========================
def compute_workload(route, d, t):
    if not route:
        return 0

    time = 0
    prev = 0

    for node in route:
        time += t[prev][node] + d[node]
        prev = node

    time += t[prev][0]  # quay về depot
    return time


# =========================
# 4. Fitness Function
# =========================
def fitness(chromosome, K, d, t):
    routes = decode(chromosome, K, d, t)
    workloads = [compute_workload(r, d, t) for r in routes]
    return max(workloads)


# =========================
# 5. Bước 3: Selection
# =========================
def selection(population, fitnesses, k=3):
    selected = random.sample(list(zip(population, fitnesses)), k)
    selected.sort(key=lambda x: x[1])
    return selected[0][0]


# =========================
# 5. Bước 4: Crossover (OX)
# =========================
def crossover(p1, p2):
    n = len(p1)
    a, b = sorted(random.sample(range(n), 2))

    child = [-1]*n
    child[a:b] = p1[a:b]

    ptr = 0
    for x in p2:
        if x not in child:
            while child[ptr] != -1:
                ptr += 1
            child[ptr] = x

    return child


# =========================
# 5. Bước 5: Mutation
# =========================
def mutate(chromosome, rate=0.1):
    if random.random() < rate:
        i, j = random.sample(range(len(chromosome)), 2)
        chromosome[i], chromosome[j] = chromosome[j], chromosome[i]


# =========================
# 5. Bước 1 → 6: GA Solver
# =========================
def GA_solver(N, K, d, t, pop_size=50, generations=200):

    # Bước 1: Khởi tạo quần thể
    population = []
    for _ in range(pop_size):
        perm = list(range(1, N+1))
        random.shuffle(perm)
        population.append(perm)

    best = None
    best_fit = float('inf')

    # Lặp qua nhiều thế hệ
    for gen in range(generations):

        # Bước 2: Đánh giá
        fitnesses = [fitness(ind, K, d, t) for ind in population]

        # cập nhật nghiệm tốt nhất
        for i in range(pop_size):
            if fitnesses[i] < best_fit:
                best_fit = fitnesses[i]
                best = population[i][:]

        new_population = []

        # Elitism (giữ nghiệm tốt nhất)
        new_population.append(best[:])

        # tạo thế hệ mới
        while len(new_population) < pop_size:

            # Bước 3: Selection
            p1 = selection(population, fitnesses)
            p2 = selection(population, fitnesses)

            # Bước 4: Crossover
            child = crossover(p1, p2)

            # Bước 5: Mutation
            mutate(child)

            new_population.append(child)

        population = new_population

        if gen % 20 == 0:
            print(f"Gen {gen}, best fitness = {best_fit}")

    return best


# =========================
# Output
# =========================
def print_solution(chromosome, K, d, t):
    routes = decode(chromosome, K, d, t)

    print(K)
    for r in routes:
        route = [0] + r + [0]
        print(len(route))
        print(*route)


In [ ]:
# =========================
# Example test
# =========================
if __name__ == "__main__":
    N = 5
    K = 2

    d = [0, 60, 80, 70, 10, 90]

    t = [
        [0, 50, 100, 60, 40, 80],
        [50, 0, 50, 40, 20, 60],
        [100, 50, 0, 50, 70, 40],
        [60, 40, 50, 0, 60, 20],
        [40, 20, 70, 60, 0, 80],
        [80, 60, 40, 20, 80, 0]
    ]

    best = GA_solver(N, K, d, t)

    print("\nBest solution:")
    print_solution(best, K, d, t)

Gen 0, best fitness = 360
Gen 20, best fitness = 360
Gen 40, best fitness = 360
Gen 60, best fitness = 360
Gen 80, best fitness = 360
Gen 100, best fitness = 360
Gen 120, best fitness = 360
Gen 140, best fitness = 360
Gen 160, best fitness = 360
Gen 180, best fitness = 360

Best solution:
2
5
0 1 2 4 0
4
0 3 5 0
